# 08 — Safe Reinforcement Learning

Standard RL maximises expected return with no constraints. Safe RL adds the requirement that the agent satisfies safety constraints throughout training and deployment — not just after convergence.

## Constrained MDP formulation

A Constrained MDP (CMDP) augments the standard MDP with a cost function C(s,a,s') and a constraint:
$$\max_\pi J(\pi) \quad \text{s.t.} \quad J_C(\pi) = \mathbb{E}\left[\sum_t \gamma^t C_t\right] \leq d$$

**Lagrangian relaxation** converts this to an unconstrained problem with a learned multiplier λ:
$$\mathcal{L}(\pi, \lambda) = J(\pi) - \lambda(J_C(\pi) - d)$$

The outer loop updates λ (dual ascent — increase λ when constraints are violated), the inner loop updates π.

## CPO and constrained PPO (narrative)

**CPO** (Constrained Policy Optimisation, Achiam et al. 2017) uses a trust-region update that simultaneously guarantees policy improvement and constraint satisfaction. It requires second-order optimisation, making it expensive.

**Constrained PPO** (simpler and more commonly used) applies the Lagrangian approach with PPO as the inner loop. This is the approach used in RLHF where the KL penalty is a soft constraint keeping the policy close to the SFT reference model.

Safe RL is active research. In deployed systems the most robust safety mechanism remains human approval gates for high-impact actions — provable guarantees on neural policies are still an open problem.

In [ ]:
# Constrained PPO (Lagrangian) on a safe-navigation task
import torch, torch.nn as nn, torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42)

class SafeNavEnv:
    def __init__(self):
        self.state_dim = 4; self.act_dim = 2
        self.goal = np.array([0.8, 0.8])
        self.forbidden_center = np.array([0.5, 0.5])
        self.forbidden_radius = 0.2
        self.dt = 0.1; self.max_steps = 100; self.reset()

    def reset(self):
        self.pos = np.array([0.0, 0.0]); self.vel = np.zeros(2); self.steps = 0
        return np.concatenate([self.pos, self.vel])

    def step(self, action):
        action = np.clip(action, -1, 1)
        self.vel = 0.8 * self.vel + 0.2 * action
        self.pos = np.clip(self.pos + self.vel * self.dt, -1, 1)
        self.steps += 1
        dist_goal = np.linalg.norm(self.pos - self.goal)
        dist_forbidden = np.linalg.norm(self.pos - self.forbidden_center)
        reward = -dist_goal
        cost = 1.0 if dist_forbidden < self.forbidden_radius else 0.0
        done = dist_goal < 0.05 or self.steps >= self.max_steps
        return np.concatenate([self.pos, self.vel]), reward, cost, done

class ActorCriticCost(nn.Module):
    def __init__(self, obs_dim=4, act_dim=2, hidden=128):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(obs_dim,hidden),nn.Tanh(),nn.Linear(hidden,hidden),nn.Tanh())
        self.actor_mean = nn.Linear(hidden, act_dim)
        self.actor_log_std = nn.Parameter(torch.zeros(act_dim))
        self.reward_critic = nn.Linear(hidden, 1)
        self.cost_critic = nn.Linear(hidden, 1)

    def forward(self, x):
        feat = self.shared(x)
        mean = torch.tanh(self.actor_mean(feat))
        std = self.actor_log_std.exp().expand_as(mean)
        return mean, std, self.reward_critic(feat).squeeze(-1), self.cost_critic(feat).squeeze(-1)

    def sample_action(self, obs):
        mean, std, rv, cv = self(obs)
        dist = torch.distributions.Normal(mean, std)
        action = dist.sample()
        lp = dist.log_prob(action).sum(-1)
        return action.clamp(-1,1), lp, rv, cv

env = SafeNavEnv()
net = ActorCriticCost()
optimizer = optim.Adam(net.parameters(), lr=3e-4)
lam = 0.0; cost_limit = 0.1; lambda_lr = 0.01

episode_rewards, episode_costs, lambda_history = [], [], []

for ep in range(500):
    obs = env.reset()
    ep_reward = ep_cost = 0
    obs_buf, act_buf, lp_buf = [], [], []
    rew_buf, cost_buf, done_buf, rval_buf, cval_buf = [], [], [], [], []

    for _ in range(100):
        obs_t = torch.FloatTensor(obs).unsqueeze(0)
        with torch.no_grad():
            action, lp, rv, cv = net.sample_action(obs_t)
        ns, r, c, done = env.step(action.squeeze(0).numpy())
        obs_buf.append(obs); act_buf.append(action.squeeze(0).numpy())
        lp_buf.append(lp.item()); rew_buf.append(r); cost_buf.append(c)
        done_buf.append(float(done)); rval_buf.append(rv.item()); cval_buf.append(cv.item())
        obs = ns; ep_reward += r; ep_cost += c
        if done: break

    episode_rewards.append(ep_reward)
    episode_costs.append(ep_cost / len(rew_buf))

    def compute_returns(rewards, values, gamma=0.99):
        G = 0; returns = []
        for r, v in zip(reversed(rewards), reversed(values)):
            G = r + gamma * G; returns.insert(0, G)
        return torch.FloatTensor(returns)

    r_returns = compute_returns(rew_buf, rval_buf)
    c_returns = compute_returns(cost_buf, cval_buf)
    obs_t = torch.FloatTensor(np.array(obs_buf))
    act_t = torch.FloatTensor(np.array(act_buf))

    mean, std, rv_pred, cv_pred = net(obs_t)
    dist = torch.distributions.Normal(mean, std)
    log_probs = dist.log_prob(act_t).sum(-1)
    old_lp = torch.FloatTensor(lp_buf)

    r_adv = r_returns - rv_pred.detach(); r_adv = (r_adv-r_adv.mean())/(r_adv.std()+1e-8)
    c_adv = c_returns - cv_pred.detach(); c_adv = (c_adv-c_adv.mean())/(c_adv.std()+1e-8)
    ratio = (log_probs - old_lp).exp()
    clipped = ratio.clamp(0.8, 1.2)
    reward_obj = torch.min(ratio*r_adv, clipped*r_adv).mean()
    cost_obj = torch.min(ratio*c_adv, clipped*c_adv).mean()
    loss = -(reward_obj - lam * cost_obj)
    loss += 0.5*(nn.functional.mse_loss(rv_pred, r_returns)+nn.functional.mse_loss(cv_pred, c_returns))
    optimizer.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 0.5); optimizer.step()

    avg_cost = np.mean(episode_costs[-20:]) if len(episode_costs)>=20 else episode_costs[-1]
    lam = max(0.0, lam + lambda_lr*(avg_cost - cost_limit))
    lambda_history.append(lam)

    if (ep+1) % 100 == 0:
        print(f"  Ep {ep+1:4d} | Reward: {np.mean(episode_rewards[-50:]):6.2f} | "
              f"Cost/step: {np.mean(episode_costs[-50:]):.3f} | lambda={lam:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
w = 30
for ax, data, title, color in [
    (axes[0], episode_rewards, 'Episode Reward', 'steelblue'),
    (axes[1], episode_costs, 'Cost per Step', 'darkorange'),
    (axes[2], lambda_history, 'Lagrange Multiplier', 'green'),
]:
    ax.plot(data, alpha=0.3, color=color)
    if len(data) >= w: ax.plot(np.convolve(data, np.ones(w)/w, 'valid'), color=color)
    ax.set_xlabel('Episode'); ax.set_title(title, fontweight='bold')
axes[1].axhline(cost_limit, color='red', linestyle='--', alpha=0.7, label=f'Limit ({cost_limit})')
axes[1].legend()
plt.tight_layout(); plt.savefig('/tmp/safe_rl.png', dpi=100, bbox_inches='tight'); plt.show()
